# 내 WAV + 타임스탬프 transcript를 Whisper Dataset으로 전처리

긴 WAV와 다음 형식의 transcript TXT를 입력받아 `openai/whisper-tiny` fine-tuning용 Hugging Face Dataset을 만듭니다.

```text
[2.270,3.400]\tG0102\tmale\t매직 데이터
[16.129,18.430]\tG0102\tmale\t방탈출 카페 창업을 한다고?
```

처리 순서:

1. 타임스탬프 TXT 파싱
2. 발화를 자르지 않고 최대 30초로 묶기
3. WAV의 해당 구간만 읽기
4. 다채널 오디오를 mono로 변환
5. 필요하면 16 kHz로 리샘플링
6. Whisper Processor로 `(80, 3000)` Log-Mel 생성
7. 한국어 transcript를 Whisper token ID로 변환
8. train/development/validation Dataset 저장

> 이 노트북은 Conv1D나 LoRA를 적용하지 않습니다. 저장되는 `input_features`는 Whisper Encoder의 Conv1D에 들어가기 직전의 Log-Mel입니다.


## 0. 패키지 설치

필요한 경우 아래 줄의 주석을 해제하고 실행한 뒤 커널을 재시작하세요.


In [14]:
# %pip install  datasets 


In [15]:
import json
import random
import re
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import librosa
import numpy as np
import soundfile as sf
from datasets import Dataset, load_from_disk
from transformers import AutoProcessor


## 1. 경로와 전처리 설정

`DATA_ITEMS`에 WAV와 transcript TXT 경로를 등록하세요. 여러 쌍을 추가할 수 있습니다.


In [16]:
# 실제 파일 경로로 변경하세요. Windows에서도 / 구분자를 사용할 수 있습니다.
DATA_ITEMS = [
    {
        'wav': Path(r'/home/lmh/project/whisper/summer_bootcamp/data/wav/A0051_S0001_0_G0101.wav'),
        'transcript': Path(r'/home/lmh/project/whisper/summer_bootcamp/data/text/A0051_S0001_0_G0101.txt'),
        # SPLIT_MODE='explicit'일 때 사용: train/development/validation
        'split': 'train',
    },
]

MODEL_ID = 'openai/whisper-small'
LANGUAGE = 'ko'
TASK = 'transcribe'
TARGET_SAMPLE_RATE = 16_000
MAX_CHUNK_SECONDS = 30.0

# 발화가 잘리지 않도록 앞뒤 문맥을 조금 포함합니다.
PRE_ROLL_SECONDS = 0.20
POST_ROLL_SECONDS = 0.20

# None이면 30초 안에서 긴 침묵도 함께 묶습니다.
# 예: 5.0이면 발화 사이 침묵이 5초를 넘을 때 새 chunk를 시작합니다.
MAX_GAP_SECONDS: Optional[float] = None
GROUP_MODE = 'pack'     # 'pack' 또는 'single'

# random: 모든 chunk를 무작위로 세 split으로 나눔
# explicit: DATA_ITEMS의 split 값을 그대로 사용
SPLIT_MODE = 'random'
DEVELOPMENT_RATIO = 0.10
VALIDATION_RATIO = 0.10
RANDOM_SEED = 42

LABEL_ENCODING = 'utf-8-sig'
OUTPUT_ROOT = Path('./my_whisper_tiny_dataset')
OVERWRITE_OUTPUT = False

# map 디버깅과 Windows/Jupyter 안정성을 위해 단일 프로세스를 사용합니다.
# 정상 실행을 확인한 뒤 환경에 맞게 2 이상으로 바꿀 수 있습니다.
NUM_PROC = None


## 2. 라벨 파싱과 30초 이하 chunk 생성


In [17]:
TIME_PATTERN = re.compile(
    r'^\[\s*(?P<start>\d+(?:\.\d+)?)\s*,\s*(?P<end>\d+(?:\.\d+)?)\s*\]$'
)


@dataclass(frozen=True)
class Segment:
    start: float
    end: float
    speaker: str
    gender: str
    text: str


@dataclass(frozen=True)
class Chunk:
    crop_start: float
    crop_end: float
    segments: Tuple[Segment, ...]

    @property
    def duration(self) -> float:
        return self.crop_end - self.crop_start

    @property
    def text(self) -> str:
        return ' '.join(segment.text.strip() for segment in self.segments).strip()


def read_timestamp_transcript(path: Path) -> List[Segment]:
    segments: List[Segment] = []

    with path.open('r', encoding=LABEL_ENCODING) as file:
        for line_number, raw_line in enumerate(file, start=1):
            line = raw_line.rstrip('\r\n')
            if not line.strip():
                continue

            fields = line.split('\t', 3)
            if len(fields) != 4:
                raise ValueError(
                    f'{path}:{line_number}: 탭으로 구분된 4개 필드가 필요합니다: {line!r}'
                )

            time_field, speaker, gender, text = fields
            match = TIME_PATTERN.fullmatch(time_field.strip())
            if match is None:
                raise ValueError(
                    f'{path}:{line_number}: 잘못된 시간 형식: {time_field!r}'
                )

            start = float(match.group('start'))
            end = float(match.group('end'))
            text = text.strip()

            if start < 0 or end <= start:
                raise ValueError(
                    f'{path}:{line_number}: 잘못된 시간: {start}, {end}'
                )
            if not text:
                raise ValueError(f'{path}:{line_number}: transcript가 비어 있습니다.')

            segments.append(
                Segment(start, end, speaker.strip(), gender.strip(), text)
            )

    segments.sort(key=lambda segment: (segment.start, segment.end))
    if not segments:
        raise ValueError(f'유효한 transcript가 없습니다: {path}')
    return segments


def make_chunk(group: Sequence[Segment], audio_duration: float) -> Chunk:
    speech_start = group[0].start
    speech_end = group[-1].end

    if speech_end > audio_duration + 0.1:
        raise ValueError(
            f'라벨 종료 {speech_end:.3f}s가 오디오 길이 {audio_duration:.3f}s를 넘습니다.'
        )

    before = min(PRE_ROLL_SECONDS, speech_start)
    after = min(POST_ROLL_SECONDS, max(0.0, audio_duration - speech_end))
    speech_duration = speech_end - speech_start

    if speech_duration > MAX_CHUNK_SECONDS:
        raise ValueError(
            f'단일 chunk의 발화 범위가 30초를 넘습니다: '
            f'{speech_start:.3f}~{speech_end:.3f}'
        )

    # 발화 자체는 30초 이하지만 margin 때문에 넘는 경우 margin을 비례 축소합니다.
    margin_budget = MAX_CHUNK_SECONDS - speech_duration
    margin_total = before + after
    if margin_total > margin_budget and margin_total > 0:
        scale = margin_budget / margin_total
        before *= scale
        after *= scale

    crop_start = speech_start - before
    crop_end = speech_end + after
    return Chunk(crop_start, crop_end, tuple(group))


def group_segments(segments: Sequence[Segment], audio_duration: float) -> List[Chunk]:
    for segment in segments:
        if segment.end - segment.start > MAX_CHUNK_SECONDS:
            raise ValueError(
                f'단일 발화가 30초를 넘습니다. 더 작은 라벨이 필요합니다: {segment}'
            )

    if GROUP_MODE == 'single':
        return [make_chunk([segment], audio_duration) for segment in segments]
    if GROUP_MODE != 'pack':
        raise ValueError(f'지원하지 않는 GROUP_MODE: {GROUP_MODE!r}')

    chunks: List[Chunk] = []
    current: List[Segment] = []

    for segment in segments:
        if not current:
            current = [segment]
            continue

        candidate_start = max(0.0, current[0].start - PRE_ROLL_SECONDS)
        candidate_end = min(audio_duration, segment.end + POST_ROLL_SECONDS)
        duration_ok = candidate_end - candidate_start <= MAX_CHUNK_SECONDS
        gap = max(0.0, segment.start - current[-1].end)
        gap_ok = MAX_GAP_SECONDS is None or gap <= MAX_GAP_SECONDS

        if duration_ok and gap_ok:
            current.append(segment)
        else:
            chunks.append(make_chunk(current, audio_duration))
            current = [segment]

    if current:
        chunks.append(make_chunk(current, audio_duration))
    return chunks


## 3. WAV·TXT에서 chunk manifest 만들기


In [18]:
def build_records(data_items: Sequence[Dict]) -> List[dict]:
    records: List[dict] = []

    for item_index, item in enumerate(data_items):
        wav_path = Path(item['wav']).expanduser().resolve()
        transcript_path = Path(item['transcript']).expanduser().resolve()
        requested_split = str(item.get('split', 'train')).strip().lower()

        if not wav_path.is_file():
            raise FileNotFoundError(f'WAV 파일이 없습니다: {wav_path}')
        if not transcript_path.is_file():
            raise FileNotFoundError(f'TXT 파일이 없습니다: {transcript_path}')
        if requested_split not in {'train', 'development', 'validation'}:
            raise ValueError(f'잘못된 split: {requested_split}')

        info = sf.info(str(wav_path))
        audio_duration = info.frames / info.samplerate
        segments = read_timestamp_transcript(transcript_path)
        chunks = group_segments(segments, audio_duration)

        print(
            f'[{item_index}] {wav_path.name}: '
            f'{audio_duration:.3f}s, segments={len(segments)}, chunks={len(chunks)}'
        )

        for chunk_index, chunk in enumerate(chunks):
            speakers = sorted({segment.speaker for segment in chunk.segments})
            genders = sorted({segment.gender for segment in chunk.segments})

            records.append(
                {
                    'file_id': f'{item_index:03d}_{wav_path.stem}_{chunk_index:05d}',
                    'source_wav': str(wav_path),
                    'source_transcript': str(transcript_path),
                    'crop_start': round(chunk.crop_start, 6),
                    'crop_end': round(chunk.crop_end, 6),
                    'duration_sec': round(chunk.duration, 6),
                    'text': chunk.text,
                    'speakers': speakers,
                    'genders': genders,
                    'requested_split': requested_split,
                    'segments': [
                        {
                            'start': segment.start,
                            'end': segment.end,
                            'relative_start': round(segment.start - chunk.crop_start, 6),
                            'relative_end': round(segment.end - chunk.crop_start, 6),
                            'speaker': segment.speaker,
                            'gender': segment.gender,
                            'text': segment.text,
                        }
                        for segment in chunk.segments
                    ],
                }
            )

    if not records:
        raise ValueError('생성된 chunk가 없습니다.')
    return records


records = build_records(DATA_ITEMS)
print('전체 chunk:', len(records))
print(json.dumps(records[0], ensure_ascii=False, indent=2))


[0] A0051_S0001_0_G0101.wav: 906.690s, segments=212, chunks=31
전체 chunk: 31
{
  "file_id": "000_A0051_S0001_0_G0101_00000",
  "source_wav": "/home/lmh/project/whisper/summer_bootcamp/data/wav/A0051_S0001_0_G0101.wav",
  "source_transcript": "/home/lmh/project/whisper/summer_bootcamp/data/text/A0051_S0001_0_G0101.txt",
  "crop_start": 5.49,
  "crop_end": 33.45,
  "duration_sec": 27.96,
  "text": "매직 데이터 그 요즘 방 탈출 카페를 좋아하는데 요즘 그래서 그 창업 쪽으로 한번 생각을 해 보고 있어요. 그 방 탈출 카페라고 해서 이제 창업 쪽도 그냥 생각을 하는데 그냥 카페를 창업하고 싶어 하는 사람들도 많고 그래서 요즘은 좀 이런 생각들도 많이 하고 있어.",
  "speakers": [
    "G0101"
  ],
  "genders": [
    "male"
  ],
  "requested_split": "train",
  "segments": [
    {
      "start": 5.69,
      "end": 6.88,
      "relative_start": 0.2,
      "relative_end": 1.39,
      "speaker": "G0101",
      "gender": "male",
      "text": "매직 데이터"
    },
    {
      "start": 7.97,
      "end": 9.46,
      "relative_start": 2.48,
      "relative_end": 3.97,
      "speaker": "G0101",
      "gender": "male",
   

## 4. Train / Development / Validation 분리

`random`은 간단한 실행 확인용입니다. 실제 성능 평가에서는 서로 다른 WAV나 화자가 split 사이에 섞이지 않도록 `explicit` 분리를 권장합니다.


In [19]:
def split_records(records: Sequence[dict]):
    if SPLIT_MODE == 'explicit':
        train = [record for record in records if record['requested_split'] == 'train']
        development = [record for record in records if record['requested_split'] == 'development']
        validation = [record for record in records if record['requested_split'] == 'validation']
    elif SPLIT_MODE == 'random':
        if len(records) < 3:
            raise ValueError('무작위 3-way split을 위해 chunk가 최소 3개 필요합니다.')

        shuffled = list(records)
        random.Random(RANDOM_SEED).shuffle(shuffled)

        n_validation = max(1, round(len(shuffled) * VALIDATION_RATIO))
        n_development = max(1, round(len(shuffled) * DEVELOPMENT_RATIO))
        if n_validation + n_development >= len(shuffled):
            raise ValueError('development/validation 비율이 너무 큽니다.')

        validation = shuffled[:n_validation]
        development = shuffled[n_validation:n_validation + n_development]
        train = shuffled[n_validation + n_development:]
    else:
        raise ValueError(f'지원하지 않는 SPLIT_MODE: {SPLIT_MODE!r}')

    if not train or not development or not validation:
        raise ValueError(
            f'각 split에 데이터가 필요합니다: '
            f'train={len(train)}, development={len(development)}, validation={len(validation)}'
        )
    return train, development, validation


train_records, development_records, validation_records = split_records(records)
print('train:', len(train_records))
print('development:', len(development_records))
print('validation:', len(validation_records))


train: 25
development: 3
validation: 3


## 5. Whisper Processor와 Hugging Face Dataset 생성


In [20]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    language=LANGUAGE,
    task=TASK,
)

EXPECTED_N_MELS = int(processor.feature_extractor.feature_size)
print('model:', MODEL_ID)
print('expected n_mels:', EXPECTED_N_MELS)

ds_train = Dataset.from_list(train_records)
ds_development = Dataset.from_list(development_records)
ds_validation = Dataset.from_list(validation_records)

print(ds_train)
print(ds_development)
print(ds_validation)


model: openai/whisper-small
expected n_mels: 80
Dataset({
    features: ['file_id', 'source_wav', 'source_transcript', 'crop_start', 'crop_end', 'duration_sec', 'text', 'speakers', 'genders', 'requested_split', 'segments'],
    num_rows: 25
})
Dataset({
    features: ['file_id', 'source_wav', 'source_transcript', 'crop_start', 'crop_end', 'duration_sec', 'text', 'speakers', 'genders', 'requested_split', 'segments'],
    num_rows: 3
})
Dataset({
    features: ['file_id', 'source_wav', 'source_transcript', 'crop_start', 'crop_end', 'duration_sec', 'text', 'speakers', 'genders', 'requested_split', 'segments'],
    num_rows: 3
})


## 6. WAV 구간 읽기와 Whisper 전처리

`prepare_example`은 한 번에 샘플 하나를 처리합니다. 따라서 `Dataset.map(..., batched=True)`를 사용하지 않습니다.


In [21]:
def read_audio_crop(
    wav_path: str,
    start_seconds: float,
    end_seconds: float,
) -> np.ndarray:
    info = sf.info(wav_path)
    original_sr = info.samplerate
    start_frame = max(0, round(start_seconds * original_sr))
    stop_frame = min(info.frames, round(end_seconds * original_sr))

    # 항상 (samples, channels) 형태로 읽은 뒤 채널 평균을 사용합니다.
    audio_2d, sample_rate = sf.read(
        wav_path,
        start=start_frame,
        stop=stop_frame,
        dtype='float32',
        always_2d=True,
    )

    if audio_2d.size == 0:
        raise ValueError(
            f'빈 오디오 구간입니다: {wav_path}, {start_seconds}~{end_seconds}'
        )

    audio = audio_2d.mean(axis=1).astype(np.float32, copy=False)
    if not np.isfinite(audio).all():
        raise ValueError(f'NaN 또는 무한대 오디오 값이 있습니다: {wav_path}')

    if sample_rate != TARGET_SAMPLE_RATE:
        audio = librosa.resample(
            audio,
            orig_sr=sample_rate,
            target_sr=TARGET_SAMPLE_RATE,
        ).astype(np.float32, copy=False)

    return audio


def prepare_example(example: dict) -> dict:
    audio = read_audio_crop(
        example['source_wav'],
        float(example['crop_start']),
        float(example['crop_end']),
    )

    feature_output = processor.feature_extractor(
        audio,
        sampling_rate=TARGET_SAMPLE_RATE,
        return_attention_mask=False,
    )
    input_features = np.asarray(
        feature_output.input_features[0], dtype=np.float32
    )

    expected_shape = (EXPECTED_N_MELS, 3000)
    if input_features.shape != expected_shape:
        raise ValueError(
            f"{example['file_id']}: Log-Mel shape={input_features.shape}, "
            f'expected={expected_shape}'
        )

    labels = processor.tokenizer(
        example['text'],
        add_special_tokens=True,
    ).input_ids

    if len(labels) > 448:
        raise ValueError(
            f"{example['file_id']}: labels={len(labels)} tokens. "
            'chunk를 더 작게 나누세요.'
        )

    example['input_features'] = input_features
    example['labels'] = labels
    example['num_audio_samples_16k'] = int(len(audio))
    example['labels_length'] = int(len(labels))
    return example


## 7. 전처리 실행


In [22]:
map_kwargs = {
    'function': prepare_example,
    'batched': False,
}
if NUM_PROC is not None:
    map_kwargs['num_proc'] = NUM_PROC

ds_train_prep = ds_train.map(
    desc='Preparing train dataset',
    **map_kwargs,
)
ds_development_prep = ds_development.map(
    desc='Preparing development dataset',
    **map_kwargs,
)
ds_validation_prep = ds_validation.map(
    desc='Preparing validation dataset',
    **map_kwargs,
)

print(ds_train_prep)
print(ds_development_prep)
print(ds_validation_prep)


Preparing validation dataset: 100%|██████████| 3/3 [00:00<00:00, 134.04 examples/s]

Dataset({
    features: ['file_id', 'source_wav', 'source_transcript', 'crop_start', 'crop_end', 'duration_sec', 'text', 'speakers', 'genders', 'requested_split', 'segments', 'input_features', 'labels', 'num_audio_samples_16k', 'labels_length'],
    num_rows: 25
})
Dataset({
    features: ['file_id', 'source_wav', 'source_transcript', 'crop_start', 'crop_end', 'duration_sec', 'text', 'speakers', 'genders', 'requested_split', 'segments', 'input_features', 'labels', 'num_audio_samples_16k', 'labels_length'],
    num_rows: 3
})
Dataset({
    features: ['file_id', 'source_wav', 'source_transcript', 'crop_start', 'crop_end', 'duration_sec', 'text', 'speakers', 'genders', 'requested_split', 'segments', 'input_features', 'labels', 'num_audio_samples_16k', 'labels_length'],
    num_rows: 3
})


## 8. 전처리 결과 검증


In [23]:
example = ds_train_prep[0]
feature_array = np.asarray(example['input_features'])

print('file_id:', example['file_id'])
print('source_wav:', example['source_wav'])
print('crop:', example['crop_start'], '~', example['crop_end'])
print('duration:', example['duration_sec'])
print('text:', example['text'])
print('input_features:', feature_array.shape, feature_array.dtype)
print('labels length:', len(example['labels']))
print('decoded label:', processor.tokenizer.decode(
    example['labels'], skip_special_tokens=True
))

assert feature_array.shape == (EXPECTED_N_MELS, 3000)
assert np.isfinite(feature_array).all()


file_id: 000_A0051_S0001_0_G0101_00005
source_wav: /home/lmh/project/whisper/summer_bootcamp/data/wav/A0051_S0001_0_G0101.wav
crop: 152.64 ~ 182.49
duration: 29.85
text: 죽은 거라고 난 얘기는 못 할 거 같아. 그래도 아직 한- 찾는 사람들도 많고 그렇지. 그래서 그 기획이라든가 좀 그쪽 그런 거를 잘 하면 괜찮은 것 같고 카페 같은 경우도 솔직히 요즘 보면은 어, 예전에 카페라고 하면은 그냥 앉아서 우리끼리 얘기하고 막 그런 거였지만 요즘은 솔직히 예전 카페보다는 인테리어도 많이 쓰고 이런 소품에도 많이
input_features: (80, 3000) float64
labels length: 107
decoded label: 죽은 거라고 난 얘기는 못 할 거 같아. 그래도 아직 한- 찾는 사람들도 많고 그렇지. 그래서 그 기획이라든가 좀 그쪽 그런 거를 잘 하면 괜찮은 것 같고 카페 같은 경우도 솔직히 요즘 보면은 어, 예전에 카페라고 하면은 그냥 앉아서 우리끼리 얘기하고 막 그런 거였지만 요즘은 솔직히 예전 카페보다는 인테리어도 많이 쓰고 이런 소품에도 많이


## 9. Dataset과 Processor 저장

`OVERWRITE_OUTPUT=True`이면 기존 출력 폴더를 삭제한 뒤 다시 생성합니다. 필요한 파일이 없는지 먼저 확인하세요.


In [24]:
if OUTPUT_ROOT.exists():
    if not OVERWRITE_OUTPUT:
        raise FileExistsError(
            f'출력 폴더가 이미 있습니다: {OUTPUT_ROOT.resolve()}\n'
            '다른 경로를 지정하거나 OVERWRITE_OUTPUT=True로 변경하세요.'
        )
    shutil.rmtree(OUTPUT_ROOT)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

train_dir = OUTPUT_ROOT / 'train'
development_dir = OUTPUT_ROOT / 'development'
validation_dir = OUTPUT_ROOT / 'validation'
processor_dir = OUTPUT_ROOT / 'processor'

ds_train_prep.save_to_disk(train_dir)
ds_development_prep.save_to_disk(development_dir)
ds_validation_prep.save_to_disk(validation_dir)
processor.save_pretrained(processor_dir)

summary = {
    'model_id': MODEL_ID,
    'language': LANGUAGE,
    'task': TASK,
    'n_mels': EXPECTED_N_MELS,
    'train_count': len(ds_train_prep),
    'development_count': len(ds_development_prep),
    'validation_count': len(ds_validation_prep),
}
with (OUTPUT_ROOT / 'dataset_summary.json').open('w', encoding='utf-8') as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

print('Saved to:', OUTPUT_ROOT.resolve())
print(json.dumps(summary, ensure_ascii=False, indent=2))


FileExistsError: 출력 폴더가 이미 있습니다: /home/lmh/project/whisper/summer_bootcamp/minhyeok/my_whisper_tiny_dataset
다른 경로를 지정하거나 OVERWRITE_OUTPUT=True로 변경하세요.

## 10. 저장 결과 다시 불러오기


In [ ]:
loaded_train = load_from_disk(OUTPUT_ROOT / 'train')
print(loaded_train)
print(loaded_train.column_names)
print('first feature shape:', np.asarray(loaded_train[0]['input_features']).shape)
print('first text:', loaded_train[0]['text'])


Dataset({
    features: ['file_id', 'source_wav', 'source_transcript', 'crop_start', 'crop_end', 'duration_sec', 'text', 'speakers', 'genders', 'requested_split', 'segments', 'input_features', 'labels', 'num_audio_samples_16k', 'labels_length'],
    num_rows: 25
})
['file_id', 'source_wav', 'source_transcript', 'crop_start', 'crop_end', 'duration_sec', 'text', 'speakers', 'genders', 'requested_split', 'segments', 'input_features', 'labels', 'num_audio_samples_16k', 'labels_length']
first feature shape: (80, 3000)
first text: 죽은 거라고 난 얘기는 못 할 거 같아. 그래도 아직 한- 찾는 사람들도 많고 그렇지. 그래서 그 기획이라든가 좀 그쪽 그런 거를 잘 하면 괜찮은 것 같고 카페 같은 경우도 솔직히 요즘 보면은 어, 예전에 카페라고 하면은 그냥 앉아서 우리끼리 얘기하고 막 그런 거였지만 요즘은 솔직히 예전 카페보다는 인테리어도 많이 쓰고 이런 소품에도 많이


## LoRA 학습 시 사용하는 열

학습 Data Collator에는 주로 다음 두 열을 전달합니다.

```text
input_features: (80, 3000) float32
labels: 가변 길이 Whisper token ID
```

배치 생성 시 `labels`를 tokenizer로 padding하고 PAD 위치를 `-100`으로 바꿉니다. 저장된 `input_features`에는 feature extractor를 다시 적용하지 않습니다.
